In [1]:
from langchain_openai import ChatOpenAI
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field

In [2]:
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=512,
)


In [3]:
class Person(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The person's age")
    city: str = Field(..., description="The person's city of residence")


In [4]:
parser = PydanticOutputParser(pydantic_object=Person)

In [5]:
template = PromptTemplate(
    template = 'Generate the name, age and city of a fictional {place} person \n {format_instructions}',
    input_variables = ['place'],
    partial_variables = {'format_instructions' : parser.get_format_instructions()}
)


In [6]:
prompt = template.invoke({'place': 'New York'})

In [10]:
print(prompt)

text='Generate the name, age and city of a fictional New York person \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "The person\'s name", "title": "Name", "type": "string"}, "age": {"description": "The person\'s age", "title": "Age", "type": "integer"}, "city": {"description": "The person\'s city of residence", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'


In [7]:
result = model.invoke(prompt)

In [9]:
print(result.content)

```json
{"name": "Jessica Thompson", "age": 34, "city": "New York"}
```


In [11]:
chain = template | model | parser

In [12]:
result2 = chain.invoke({'place': 'Village hidden in the leaf'})

In [14]:
print(result2)

name='Kaito' age=27 city='Konoha'
